##### ============================================================
# README
##### ============================================================
## Nome: Decadal Median Surfaces by AOI
#
### Descrição:
#### Este script lê as superfícies TIFF armazenadas por ano dentro da pasta 04_SUPERFICIES de cada AOI, agrupa os rasters por década e calcula:
* A superfície mediana pixel a pixel para cada década
* A imagem de valid fraction por pixel, indicando a fração da série decadal que contribuiu em cada pixel
#
### Décadas consideradas:
- 1985-1989
- 1990-1999
- 2000-2009
- 2010-2019
- 2020-2024
#
### Saídas:
- 1 GeoTIFF de mediana por AOI e década
- 1 GeoTIFF de valid fraction por AOI e década
#
### Observação:
#### A valid fraction varia de 0 a 1:
- 1.0 = pixel válido em todas as imagens da década
- 0.2 = pixel válido em 20% das imagens da década
#
#### O script também gera links/download direto no Colab para os arquivos finais.
#### ======================================================


In [1]:
# ============================================
# 1. Instalar bibliotecas
# ============================================
!pip install -q rasterio numpy

# ============================================
# 2. Importar bibliotecas
# ============================================
import os
import glob
import math
import shutil
import zipfile
import numpy as np
import rasterio

from collections import Counter
from rasterio.crs import CRS
from rasterio.warp import reproject, Resampling, transform_bounds
from rasterio.transform import from_origin
from google.colab import drive, files

In [2]:
# ============================================
# 3. Montar Google Drive
# ============================================
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:
# ============================================
# 4. Configurações principais
# ============================================
ROOT = "/content/drive/Shareddrives/Batimetrias_Babitonga"

AOIS = [
    "01_AOI01",
    "02_AOI02",
    "03_AOI03"
]

# pasta de saída no drive
OUTPUT_DIR = os.path.join(ROOT, "_DECADAL_MEDIAN_SURFACES")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# CRS alvo
TARGET_CRS = CRS.from_epsg(4326)

# Décadas
DECADES = {
    "1985_1989": (1985, 1989),
    "1990_1999": (1990, 1999),
    "2000_2009": (2000, 2009),
    "2010_2019": (2010, 2019),
    "2020_2024": (2020, 2024),
}

# ============================================
# 5. Funções auxiliares
# ============================================
def find_yearly_tifs(aoi_dir):
    """
    Retorna dict {ano_int: caminho_tif}
    Procura em 04_SUPERFICIES/ANO/
    Se houver mais de um tif no ano, usa o primeiro encontrado.
    """
    surf_dir = os.path.join(aoi_dir, "04_SUPERFICIES")
    yearly = {}

    if not os.path.isdir(surf_dir):
        return yearly

    year_folders = sorted(os.listdir(surf_dir))
    for year in year_folders:
        year_path = os.path.join(surf_dir, year)
        if not os.path.isdir(year_path):
            continue

        try:
            year_int = int(year)
        except:
            continue

        tif_list = sorted(
            glob.glob(os.path.join(year_path, "*.tif")) +
            glob.glob(os.path.join(year_path, "*.tiff"))
        )

        if len(tif_list) == 0:
            tif_list = sorted(
                glob.glob(os.path.join(year_path, "**", "*.tif"), recursive=True) +
                glob.glob(os.path.join(year_path, "**", "*.tiff"), recursive=True)
            )

        if tif_list:
            yearly[year_int] = tif_list[0]

    return yearly


def get_dominant_crs(raster_paths):
    """
    Retorna o CRS dominante entre os rasters.
    Se nenhum raster tiver CRS, assume EPSG:4326.
    """
    crs_list = []

    for path in raster_paths:
        try:
            with rasterio.open(path) as src:
                if src.crs is not None:
                    crs_list.append(src.crs.to_string())
        except Exception as e:
            print(f"[AVISO] Erro ao ler CRS de {path}: {e}")

    if not crs_list:
        print("  [AVISO] Nenhum raster com CRS definido. Assumindo EPSG:4326")
        return CRS.from_epsg(4326)

    dominant = Counter(crs_list).most_common(1)[0][0]
    return CRS.from_string(dominant)


def build_common_grid_union(raster_paths, target_crs):
    """
    Cria grade comum usando:
    - CRS alvo
    - resolução do primeiro raster disponível
    - união total das extensões
    """
    if not raster_paths:
        raise ValueError("Nenhum raster fornecido.")

    xres = None
    yres = None
    all_bounds = []

    for path in raster_paths:
        with rasterio.open(path) as src:
            if xres is None:
                xres = abs(src.transform.a)
                yres = abs(src.transform.e)

            src_crs = src.crs if src.crs is not None else target_crs

            if src_crs == target_crs:
                b = src.bounds
                all_bounds.append((b.left, b.bottom, b.right, b.top))
            else:
                b = transform_bounds(src_crs, target_crs, *src.bounds, densify_pts=21)
                all_bounds.append(b)

    if xres is None or yres is None:
        raise ValueError("Não foi possível determinar a resolução dos rasters.")

    minx = min(b[0] for b in all_bounds)
    miny = min(b[1] for b in all_bounds)
    maxx = max(b[2] for b in all_bounds)
    maxy = max(b[3] for b in all_bounds)

    width = int(math.ceil((maxx - minx) / xres))
    height = int(math.ceil((maxy - miny) / yres))

    transform = from_origin(minx, maxy, xres, yres)

    return {
        "crs": target_crs,
        "transform": transform,
        "width": width,
        "height": height,
        "xres": xres,
        "yres": yres
    }


def read_to_common_grid(path, common_grid, fallback_crs=None):
    """
    Lê e reprojeta um raster para a grade comum.
    """
    with rasterio.open(path) as src:
        arr = src.read(1).astype("float32")
        src_nodata = src.nodata
        src_crs = src.crs if src.crs is not None else fallback_crs

        if src_crs is None:
            raise ValueError(f"Raster sem CRS e sem fallback definido: {path}")

        if src_nodata is not None:
            arr[arr == src_nodata] = np.nan

        dst_arr = np.full(
            (common_grid["height"], common_grid["width"]),
            np.nan,
            dtype="float32"
        )

        reproject(
            source=arr,
            destination=dst_arr,
            src_transform=src.transform,
            src_crs=src_crs,
            dst_transform=common_grid["transform"],
            dst_crs=common_grid["crs"],
            src_nodata=np.nan,
            dst_nodata=np.nan,
            resampling=Resampling.bilinear
        )

    return dst_arr


def compute_decadal_products(raster_paths, out_median, out_fraction):
    """
    Calcula:
    - mediana pixel a pixel
    - valid fraction por pixel
    """
    target_crs = get_dominant_crs(raster_paths)
    print(f"  CRS de referência: {target_crs}")

    common_grid = build_common_grid_union(raster_paths, target_crs)

    arrays = []
    used_paths = []

    for path in raster_paths:
        try:
            with rasterio.open(path) as src:
                if src.crs is None:
                    print(f"  [AVISO] CRS ausente em {os.path.basename(path)} -> assumindo {target_crs}")

            arr = read_to_common_grid(path, common_grid, fallback_crs=target_crs)
            arrays.append(arr)
            used_paths.append(path)

        except Exception as e:
            print(f"  [AVISO] Raster ignorado: {os.path.basename(path)} | motivo: {e}")

    if not arrays:
        raise ValueError("Nenhum raster pôde ser processado.")

    stack = np.stack(arrays, axis=0)   # (n, rows, cols)

    # mediana
    median_arr = np.nanmedian(stack, axis=0).astype("float32")

    # valid fraction
    valid_count = np.sum(~np.isnan(stack), axis=0).astype("float32")
    n_total = float(stack.shape[0])
    fraction_arr = (valid_count / n_total).astype("float32")

    # perfis de saída
    out_nodata = -9999.0

    median_out = median_arr.copy()
    median_out[np.isnan(median_out)] = out_nodata

    fraction_out = fraction_arr.copy()
    fraction_out[np.isnan(fraction_out)] = out_nodata

    profile = {
        "driver": "GTiff",
        "height": common_grid["height"],
        "width": common_grid["width"],
        "count": 1,
        "crs": common_grid["crs"],
        "transform": common_grid["transform"],
        "compress": "lzw",
    }

    profile_median = profile.copy()
    profile_median.update(dtype="float32", nodata=out_nodata)

    profile_fraction = profile.copy()
    profile_fraction.update(dtype="float32", nodata=out_nodata)

    with rasterio.open(out_median, "w", **profile_median) as dst:
        dst.write(median_out, 1)

    with rasterio.open(out_fraction, "w", **profile_fraction) as dst:
        dst.write(fraction_out, 1)

    print(f"  {len(used_paths)} rasters usados.")


def zip_and_download(file_paths, zip_name):
    """
    Compacta arquivos e inicia download no Colab.
    """
    zip_path = f"/content/{zip_name}.zip"

    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
        for fp in file_paths:
            if os.path.exists(fp):
                z.write(fp, arcname=os.path.basename(fp))

    files.download(zip_path)


In [4]:
# ============================================
# 6. Processamento principal
# ============================================
download_files = []

for aoi in AOIS:
    aoi_dir = os.path.join(ROOT, aoi)

    if not os.path.isdir(aoi_dir):
        print(f"[AVISO] AOI não encontrada: {aoi_dir}")
        continue

    yearly_tifs = find_yearly_tifs(aoi_dir)

    if not yearly_tifs:
        print(f"[AVISO] Nenhum TIFF encontrado para {aoi}")
        continue

    print(f"\n==============================")
    print(f"Processando {aoi}")
    print(f"{len(yearly_tifs)} rasters anuais encontrados")
    print(f"==============================")

    aoi_out_dir = os.path.join(OUTPUT_DIR, aoi)
    os.makedirs(aoi_out_dir, exist_ok=True)

    for decade_name, (start_year, end_year) in DECADES.items():
        decadal_paths = [
            path for year, path in yearly_tifs.items()
            if start_year <= year <= end_year
        ]

        if len(decadal_paths) == 0:
            print(f"\n{aoi} | {decade_name}: sem rasters disponíveis")
            continue

        print(f"\n{aoi} | {decade_name}: {len(decadal_paths)} rasters")

        out_median = os.path.join(aoi_out_dir, f"{aoi}_{decade_name}_median.tif")
        out_fraction = os.path.join(aoi_out_dir, f"{aoi}_{decade_name}_valid_fraction.tif")

        try:
            compute_decadal_products(decadal_paths, out_median, out_fraction)
            print(f"[OK] Mediana salva em: {out_median}")
            print(f"[OK] Valid fraction salva em: {out_fraction}")

            download_files.extend([out_median, out_fraction])

        except Exception as e:
            print(f"[ERRO] {aoi} | {decade_name}: {e}")

print("\nProcessamento concluído.")

# ============================================
# 7. Download direto para o computador
# ============================================
if download_files:
    print("\nPreparando download dos arquivos...")
    zip_and_download(download_files, "decadal_median_surfaces_outputs")
else:
    print("\nNenhum arquivo foi gerado para download.")


Processando 01_AOI01
33 rasters anuais encontrados

01_AOI01 | 1985_1989: 4 rasters
  CRS de referência: EPSG:4326


/tmp/ipykernel_663/3745296410.py:215: RuntimeWarning: All-NaN slice encountered
  median_arr = np.nanmedian(stack, axis=0).astype("float32")


  4 rasters usados.
[OK] Mediana salva em: /content/drive/Shareddrives/Batimetrias_Babitonga/_DECADAL_MEDIAN_SURFACES/01_AOI01/01_AOI01_1985_1989_median.tif
[OK] Valid fraction salva em: /content/drive/Shareddrives/Batimetrias_Babitonga/_DECADAL_MEDIAN_SURFACES/01_AOI01/01_AOI01_1985_1989_valid_fraction.tif

01_AOI01 | 1990_1999: 7 rasters
  CRS de referência: EPSG:4326
  [AVISO] CRS ausente em dup_delta_1995.tif -> assumindo EPSG:4326
  [AVISO] CRS ausente em sup_delta_1996.tif -> assumindo EPSG:4326
  [AVISO] CRS ausente em sup_delta_1997.tif -> assumindo EPSG:4326
  [AVISO] CRS ausente em sup_delta_1999.tif -> assumindo EPSG:4326


/tmp/ipykernel_663/3745296410.py:215: RuntimeWarning: All-NaN slice encountered
  median_arr = np.nanmedian(stack, axis=0).astype("float32")


  7 rasters usados.
[OK] Mediana salva em: /content/drive/Shareddrives/Batimetrias_Babitonga/_DECADAL_MEDIAN_SURFACES/01_AOI01/01_AOI01_1990_1999_median.tif
[OK] Valid fraction salva em: /content/drive/Shareddrives/Batimetrias_Babitonga/_DECADAL_MEDIAN_SURFACES/01_AOI01/01_AOI01_1990_1999_valid_fraction.tif

01_AOI01 | 2000_2009: 7 rasters
  [AVISO] Nenhum raster com CRS definido. Assumindo EPSG:4326
  CRS de referência: EPSG:4326
  [AVISO] CRS ausente em sup_delta_2000.tif -> assumindo EPSG:4326
  [AVISO] CRS ausente em sup_delta_2001.tif -> assumindo EPSG:4326
  [AVISO] CRS ausente em sup_delta_2004.tif -> assumindo EPSG:4326
  [AVISO] CRS ausente em sup_delta_2005.tif -> assumindo EPSG:4326
  [AVISO] CRS ausente em sup_delta_2006.tif -> assumindo EPSG:4326
  [AVISO] CRS ausente em sup_delta_2007.tif -> assumindo EPSG:4326
  [AVISO] CRS ausente em sup_delta_2009.tif -> assumindo EPSG:4326


...

/tmp/ipykernel_663/3745296410.py:215: RuntimeWarning: All-NaN slice encountered
  median_arr = np.nanmedian(stack, axis=0).astype("float32")


  CRS de referência: EPSG:4326
  [AVISO] CRS ausente em sup_bsul_2020.tif -> assumindo EPSG:4326
  [AVISO] CRS ausente em sup_bsul_2021.tif -> assumindo EPSG:4326
  5 rasters usados.
[OK] Mediana salva em: /content/drive/Shareddrives/Batimetrias_Babitonga/_DECADAL_MEDIAN_SURFACES/03_AOI03/03_AOI03_2020_2024_median.tif
[OK] Valid fraction salva em: /content/drive/Shareddrives/Batimetrias_Babitonga/_DECADAL_MEDIAN_SURFACES/03_AOI03/03_AOI03_2020_2024_valid_fraction.tif

Processamento concluído.

Preparando download dos arquivos...


/tmp/ipykernel_663/3745296410.py:215: RuntimeWarning: All-NaN slice encountered
  median_arr = np.nanmedian(stack, axis=0).astype("float32")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>